# ES4304 Tutorial 3.2 — Homework Assignment (Google Colab)

Complete the
`TODO` sections to generate the interferogram, then run the correlation
coefficient and confusion matrix sections, and code the accuracy statistics
yourself.


## Assignment instructions (from the handout, Section 5)

You are required to complete the sections below to generate the
interferogram and threshold the correlation coefficients:

1. Using the same dataset, calculate the interferogram.
2. Acquire the phase difference from the resulting interferogram.
3. Save the phase difference as a GeoTIFF to visualise in QGIS.
   **Only two lines are needed to retrieve the phase difference. You can use AI to help you fill up the rows of code first. If you are stuck,
   please email for help. Run the notebook section by section!**
4. Run the **Calculate Correlation Co-efficient** section to generate the
   correlation coefficients (may take a few minutes). Load `CC.tif` into
   QGIS.
5. Crop both `20190716.wprobs.geo.tif` and `CC.tif` in QGIS so they cover
   the same area (see the handout Annex — Clip Raster by Extent, using
   `437430.7221,462557.7093,3945328.2161,3966344.5672 [EPSG:32611]`).
6. Run the **Calculate Confusion
   Matrix** section with the given threshold, then code the precision,
   recall, F1, and overall accuracy yourself.
  

## 0. Install dependencies

In [1]:
!pip install -q rasterio

## 1. Get your `Raw_data.zip` onto this Colab runtime

Pick whichever is fastest for you in the form below, then run the cell.

- **`url`** (fastest, recommended for large zips — e.g. ~1GB+): host
  `Raw_data.zip` somewhere with a direct download link (your own cloud
  storage, an NTULearn direct link, S3, etc.) and paste it into
  `direct_download_url`. This downloads server-to-server with no browser
  relay overhead — much faster than the other two options.
- **`drive`**: upload `Raw_data.zip` to your Google Drive first (the Drive
  web page's uploader is chunked/resumable and noticeably faster than
  `upload` below for large files), then this mounts your Drive (one-click
  consent, not a full sign-in) and reads the zip from `drive_zip_path`.
  Convenient if you'll reconnect across multiple sessions, since you only
  upload once.
- **`upload`**: a plain browser file picker
  (`google.colab.files.upload()`). No Google Drive involved at all, but it
  base64-encodes the file and relays it through the notebook's comm
  channel, which gets slow for files much larger than ~100-200MB.

In [2]:
import zipfile

direct_download_url = "https://eyes-on-earth-tutorial-resources.s3.ap-southeast-1.amazonaws.com/tutorial_3/Raw_data.zip"
zip_name = "Raw_data.zip"

!wget -q --show-progress -O "{zip_name}" "{direct_download_url}"

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(".")

print("Extracted. .h5 files found:")
!find . -iname "*.h5"

Raw_data.zip        100%[===================>]   1.50G  13.1MB/s    in 2m 2s   
Extracted. .h5 files found:
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151219-IW2_20190716T135210Z_20240430T020420Z_S1A_VV_v1.1.h5
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151217-IW2_20190704T135203Z_20240429T174024Z_S1A_VV_v1.1.h5
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151217-IW2_20190716T135204Z_20240430T020420Z_S1A_VV_v1.1.h5
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151218-IW2_20190704T135206Z_20240429T174024Z_S1A_VV_v1.1.h5
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151218-IW2_20190716T135207Z_20240430T020420Z_S1A_VV_v1.1.h5
./__MACOSX/Raw_data/._OPERA_L2_CSLC-S1_T071-151219-IW2_20190704T135209Z_20240429T174024Z_S1A_VV_v1.1.h5
./Raw_data/OPERA_L2_CSLC-S1_T071-151217-IW2_20190704T135203Z_20240429T174024Z_S1A_VV_v1.1.h5
./Raw_data/OPERA_L2_CSLC-S1_T071-151217-IW2_20190716T135204Z_20240430T020420Z_S1A_VV_v1.1.h5
./Raw_data/OPERA_L2_CSLC-S1_T071-151218-IW2_20190716T135207Z_20240430T020420Z_S1A_

If the file listing above is empty, your zip probably extracted with an
extra nested folder (e.g. `Raw_data/Raw_data/*.h5`). Either re-zip so the
`.h5` files sit directly inside `Raw_data/`, or edit the `directory`
variable in the next section to match wherever the `.h5` files ended up.

## 2. Setup: helper functions

In [3]:
# Setup: helper functions (Python port of Functions/read_cslc_function.m and
# geotiffwrite, condensed for a self-contained Colab notebook)
import os
import glob
import numpy as np
import h5py
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS


def read_cslc(directory, cslc_filename):
    """Read one OPERA CSLC-S1 HDF5 file. Returns (complex_matrix, transform,
    burstid, datestring, epsg_code)."""
    filepath = os.path.join(directory, cslc_filename)
    with h5py.File(filepath, "r") as f:
        vv = f["/data/VV"][()]
        if np.iscomplexobj(vv):
            complex_matrix = vv
        else:
            complex_matrix = vv["r"] + 1j * vv["i"]

        epsg_code = int(f["/data/projection"][()])
        x_coordinates = f["/data/x_coordinates"][()]
        y_coordinates = f["/data/y_coordinates"][()]

        burstid = f["/identification/burst_id"][()]
        if isinstance(burstid, bytes):
            burstid = burstid.decode()

        datestring = f["/metadata/calibration_information/azimuth_time"][()]
        if isinstance(datestring, bytes):
            datestring = datestring.decode()
        datestring = datestring[:10]

    x_min, x_max = float(np.min(x_coordinates)), float(np.max(x_coordinates))
    y_min, y_max = float(np.min(y_coordinates)), float(np.max(y_coordinates))
    transform = from_bounds(x_min, y_min, x_max, y_max, width=len(x_coordinates), height=len(y_coordinates))

    return complex_matrix, transform, burstid, datestring, epsg_code


def write_geotiff(path, array, transform, epsg_code):
    """Write a single-band array to a GeoTIFF (equivalent of MATLAB's geotiffwrite)."""
    with rasterio.open(
        path, "w", driver="GTiff",
        height=array.shape[0], width=array.shape[1],
        count=1, dtype=array.dtype if array.dtype != np.complex128 else "float32",
        crs=CRS.from_epsg(int(epsg_code)),
        transform=transform,
    ) as dst:
        dst.write(array, 1)


print("Helper functions ready: read_cslc(), write_geotiff()")

Helper functions ready: read_cslc(), write_geotiff()


In [4]:
# Setup: additional helper functions needed for Tutorial 3.2 (Python port of
# Functions/make_cc.m, windowit.m, adjustMapReference.m, make_confusion_matrix.m)
from scipy.signal import get_window
from scipy.ndimage import zoom
from rasterio.transform import array_bounds
from rasterio.windows import Window


def windowit(m, wname="triang", *args):
    m = np.asarray(m)
    ny, nx = m.shape
    if wname is None or wname == "none":
        w = np.ones_like(m, dtype=float)
    else:
        wx = get_window((wname, *args) if args else wname, nx)
        wy = get_window((wname, *args) if args else wname, ny)
        w = np.outer(wy, wx)
    mw = m * w
    wscale = w.size / np.sum(w ** 2)
    return mw, w, wscale


def adjust_map_reference(transform, shape, wx, wy, cc_shape):
    """New transform for a multilooked (downsampled) raster."""
    height, width = shape
    min_x, min_y, max_x, max_y = array_bounds(height, width, transform)

    cell_x = transform.a
    cell_y = abs(transform.e)

    new_min_x = min_x + ((wx - 1) / 2) * cell_x
    new_max_x = max_x - ((wx - 1) / 2) * cell_x
    new_min_y = min_y + ((wy - 1) / 2) * cell_y
    new_max_y = max_y - ((wy - 1) / 2) * cell_y

    new_height, new_width = cc_shape
    return from_bounds(new_min_x, new_min_y, new_max_x, new_max_y, new_width, new_height)


def make_cc(amp1_path, phs1_path, amp2_path, phs2_path, wx, wy, out_path, transform, epsg_code, rows_per_chunk=200):
    """Correlation coefficient (coherence) map, non-overlapping wy x wx
    windows. Streams row-chunks straight off the Amplitude/Phase GeoTIFFs and
    writes the result straight to out_path -- it never holds the
    full-resolution rasters in memory (a naive whole-array version ends up
    with several full-burst-sized temporaries alive at once, which is enough
    to crash memory-constrained runtimes on a full-size Sentinel-1 burst).
    Lower rows_per_chunk (e.g. to 50) if this still runs out of memory."""
    if wx % 2 != 1 or wy % 2 != 1:
        raise ValueError("wx and wy should be odd numbers.")

    with rasterio.open(amp1_path) as src:
        height, width = src.shape

    n_rows = height // wy
    n_cols = width // wx
    width_trunc = n_cols * wx
    total_rows_used = n_rows * wy

    wgt, _, _ = windowit(np.ones((wy, wx)), "triang")
    wgt = wgt.reshape(1, wy, 1, wx)

    rows_per_chunk = max(wy, (rows_per_chunk // wy) * wy)

    out_transform = adjust_map_reference(transform, (height, width), wx, wy, (n_rows, n_cols))
    profile = {
        "driver": "GTiff", "height": n_rows, "width": n_cols, "count": 1,
        "dtype": "float32", "crs": CRS.from_epsg(int(epsg_code)), "transform": out_transform,
    }

    with rasterio.open(amp1_path) as amp1_src, rasterio.open(phs1_path) as phs1_src, \
            rasterio.open(amp2_path) as amp2_src, rasterio.open(phs2_path) as phs2_src, \
            rasterio.open(out_path, "w", **profile) as dst:

        cc_min, cc_max = np.inf, -np.inf

        for row_start in range(0, total_rows_used, rows_per_chunk):
            rows_to_read = min(rows_per_chunk, total_rows_used - row_start)
            window = Window(0, row_start, width_trunc, rows_to_read)

            amp1 = amp1_src.read(1, window=window)
            phs1 = phs1_src.read(1, window=window)
            file1 = (amp1.astype(np.float64) ** (1.0 / 0.3)) * np.exp(1j * phs1)
            del amp1, phs1

            amp2 = amp2_src.read(1, window=window)
            phs2 = phs2_src.read(1, window=window)
            file2 = (amp2.astype(np.float64) ** (1.0 / 0.3)) * np.exp(1j * phs2)
            del amp2, phs2

            chunk_n_rows = rows_to_read // wy

            ifg = (file1 * np.conj(file2)).reshape(chunk_n_rows, wy, n_cols, wx)
            p1 = (np.abs(file1) ** 2).reshape(chunk_n_rows, wy, n_cols, wx)
            p2 = (np.abs(file2) ** 2).reshape(chunk_n_rows, wy, n_cols, wx)
            del file1, file2

            # The numerator and the denominator must carry the SAME window
            # weights. The previous version formed a1s = |c1| * wgt and then
            # summed a1s ** 2, weighting the denominator by wgt ** 2 while the
            # numerator carried only wgt. Since 0 <= wgt <= 1 that understates
            # the denominator (by up to 1.45x for an 11x5 triangular window) and
            # lets the ratio exceed 1 - observed up to ~3 on coherent speckle and
            # ~13 over bright scatterers. With both sides weighted by wgt the
            # weighted Cauchy-Schwarz inequality applies and cc is bounded by 1.
            numer = np.abs((ifg * wgt).sum(axis=(1, 3)))
            denom = np.sqrt((p1 * wgt).sum(axis=(1, 3)) * (p2 * wgt).sum(axis=(1, 3)))
            del ifg, p1, p2

            cc_chunk = np.divide(numer, denom, out=np.zeros_like(numer), where=denom != 0)
            np.clip(cc_chunk, 0.0, 1.0, out=cc_chunk)   # guard against float round-off
            cc_chunk = cc_chunk.astype(np.float32)
            cc_min = min(cc_min, float(cc_chunk.min()))
            cc_max = max(cc_max, float(cc_chunk.max()))

            dst.write(cc_chunk, 1, window=Window(0, row_start // wy, n_cols, chunk_n_rows))

    if np.isfinite(cc_min):
        print(f"Completed - coherence range: {cc_min:.4f} to {cc_max:.4f} "
              f"(must lie within 0 to 1)")
    else:
        print("Completed - no data written")
    return out_path


def _resize_nearest(array, out_shape):
    zoom_factors = (out_shape[0] / array.shape[0], out_shape[1] / array.shape[1])
    return zoom(array, zoom_factors, order=0)


def make_confusion_matrix(wprobs_matrix, cc_matrix, wprobs_threshold, cc_threshold):
    """wprobs > threshold is positive (higher = more change). cc < threshold is
    positive (lower correlation = more change)."""
    wprobs_matrix = wprobs_matrix.astype(float).copy()
    cc_matrix = cc_matrix.astype(float).copy()

    wprobs_matrix[wprobs_matrix == 0] = np.nan
    cc_matrix[cc_matrix == 0] = np.nan

    if wprobs_matrix.shape[0] < cc_matrix.shape[0] and wprobs_matrix.shape[1] < cc_matrix.shape[1]:
        cc_matrix = _resize_nearest(cc_matrix, wprobs_matrix.shape)
    elif wprobs_matrix.shape[0] > cc_matrix.shape[0] and wprobs_matrix.shape[1] > cc_matrix.shape[1]:
        wprobs_matrix = _resize_nearest(wprobs_matrix, cc_matrix.shape)

    binary1 = wprobs_matrix > wprobs_threshold
    binary2 = cc_matrix < cc_threshold

    TP = int(np.sum((binary1 == 1) & (binary2 == 1)))
    TN = int(np.sum((binary1 == 0) & (binary2 == 0)))
    FP = int(np.sum((binary1 == 0) & (binary2 == 1)))
    FN = int(np.sum((binary1 == 1) & (binary2 == 0)))

    return TP, TN, FP, FN


def load_complex_from_geotiffs(amp_path, phs_path):
    """Reconstruct a file's complex CSLC array from its saved Amplitude/Phase
    GeoTIFFs (inverting A = |C|**0.3 and PHS = angle(C); exact to float64
    precision). Lets later cells get back complex data for a specific file
    without needing the ingest loop's in-memory results to still exist —
    which is what makes it possible to jump straight to the interferogram /
    correlation-coefficient cells after a crash, as long as ./Amplitudes/
    and ./Phases/ are still on disk."""
    with rasterio.open(amp_path) as src:
        amp = src.read(1)
    with rasterio.open(phs_path) as src:
        phs = src.read(1)
    magnitude = amp.astype(np.float64) ** (1.0 / 0.3)
    return magnitude * np.exp(1j * phs)


print("Helper functions ready: make_cc(), windowit(), adjust_map_reference(), make_confusion_matrix(), load_complex_from_geotiffs()")

Helper functions ready: make_cc(), windowit(), adjust_map_reference(), make_confusion_matrix(), load_complex_from_geotiffs()


## 3. Read the CSLC files and generate Amplitude & Phase GeoTIFFs

This cell reads every CSLC .h5 file in Raw_data/, one at a time, and for each file:

1. Reads the complex SAR data, along with its acquisition date, burst ID, and coordinate reference system.
2. Computes the Amplitude (the magnitude of the complex number) and the Phase (the angle of the complex number).
3. Saves the Amplitude and Phase as separate GeoTIFF files in ./Amplitudes/ and ./Phases/, ready to view in QGIS or to be reused by later sections of this notebook.

It also saves the file list, dates, burst IDs, and coordinate info to checkpoint_metadata.pkl. If you re-run this cell later (e.g. after reconnecting), it loads that checkpoint instead of re-processing every file from scratch, as long as checkpoint_metadata.pkl, Amplitudes/, and Phases/ are still there.

In [5]:
# Read every CSLC .h5 file in Raw_data/, and save its Amplitude and Phase as GeoTIFFs
import pickle

output_dir_amp = "./Amplitudes/"
output_dir_phs = "./Phases/"
os.makedirs(output_dir_amp, exist_ok=True)
os.makedirs(output_dir_phs, exist_ok=True)

metadata_path = "./checkpoint_metadata.pkl"

if os.path.exists(metadata_path):
    print(f"Found {metadata_path} -- loading metadata instead of re-reading the .h5 files")
    with open(metadata_path, "rb") as f:
        files, burstid, datestring, epsg, R = pickle.load(f)
    num_files = len(files)
else:
    directory = "./Raw_data/"
    files = sorted(glob.glob(os.path.join(directory, "*.h5")))
    num_files = len(files)
    print(f"Found {num_files} .h5 files in {directory}")

    burstid = [None] * num_files
    datestring = [None] * num_files
    epsg = [None] * num_files
    R = [None] * num_files       # transform (map referencing) for each file

    for k, filepath in enumerate(files):
        filename = os.path.basename(filepath)

        # read this file's raw data: complex number, transform (map info),
        # date, coordinate reference system
        C_k, R[k], burstid[k], datestring[k], epsg[k] = read_cslc(directory, filename)

        # magnitude of the complex number (amplitude), raised to power 0.3
        # for better visualisation
        amp = np.abs(C_k) ** 0.3

        # phase of the complex number
        phs = np.angle(C_k)

        # save amplitude and phase as GeoTIFFs
        write_geotiff(os.path.join(output_dir_phs, f"PHS{k + 1}_{datestring[k]}.tif"), phs, R[k], epsg[k])
        write_geotiff(os.path.join(output_dir_amp, f"A{k + 1}_{datestring[k]}.tif"), amp, R[k], epsg[k])
        del C_k, amp, phs  # free this file's arrays before moving to the next one

    with open(metadata_path, "wb") as f:
        pickle.dump((files, burstid, datestring, epsg, R), f)
    print(f"Saved checkpoint metadata to {metadata_path}")

print("Done — Amplitude and Phase GeoTIFFs are in ./Amplitudes/ and ./Phases/")

Found 6 .h5 files in ./Raw_data/
Saved checkpoint metadata to ./checkpoint_metadata.pkl
Done — Amplitude and Phase GeoTIFFs are in ./Amplitudes/ and ./Phases/


In [9]:
# Make lightweight PNG previews of the Amplitude and Phase GeoTIFFs
# (for anyone who can't download the full-size .tif files)
import glob, zipfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PNG_MAX_PIXELS = 2000        # longest side of each preview image


def read_tif_preview(path, max_pixels=PNG_MAX_PIXELS):
    """Read a GeoTIFF at reduced size; return (array, extent) for plotting."""
    try:
        import rasterio
        with rasterio.open(path) as src:
            step = max(1, int(np.ceil(max(src.height, src.width) / max_pixels)))
            out_shape = (int(np.ceil(src.height / step)), int(np.ceil(src.width / step)))
            arr = src.read(1, out_shape=out_shape).astype(np.float32)
            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan
            b = src.bounds
            extent = [b.left, b.right, b.bottom, b.top]
    except ImportError:                       # fall back to GDAL if rasterio isn't installed
        from osgeo import gdal
        ds = gdal.Open(path)
        step = max(1, int(np.ceil(max(ds.RasterYSize, ds.RasterXSize) / max_pixels)))
        arr = ds.GetRasterBand(1).ReadAsArray()[::step, ::step].astype(np.float32)
        gt = ds.GetGeoTransform()
        extent = [gt[0], gt[0] + gt[1] * ds.RasterXSize,
                  gt[3] + gt[5] * ds.RasterYSize, gt[3]]
        ds = None
    return arr, extent


def save_png(path, kind):
    arr, extent = read_tif_preview(path)
    fig, ax = plt.subplots(figsize=(8, 7))
    if kind == "amplitude":
        lo, hi = np.nanpercentile(arr, [2, 98])          # stretch so detail is visible
        im = ax.imshow(arr, cmap="gray", vmin=lo, vmax=hi, extent=extent, interpolation="nearest")
        label = "Amplitude$^{0.3}$ (stretched 2–98%)"
    else:
        im = ax.imshow(arr, cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent, interpolation="nearest")
        label = "Phase (radians)"
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(label)
    if kind == "phase":
        cbar.set_ticks([-np.pi, 0, np.pi])
        cbar.set_ticklabels(["−π", "0", "π"])
    ax.set_title(os.path.basename(path).replace(".tif", ""))
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    fig.tight_layout()
    png_path = path.replace(".tif", ".png")
    fig.savefig(png_path, dpi=150)
    plt.close(fig)
    return png_path


pngs = []
for tif in sorted(glob.glob(os.path.join(output_dir_amp, "*.tif"))):
    pngs.append(save_png(tif, "amplitude"))
for tif in sorted(glob.glob(os.path.join(output_dir_phs, "*.tif"))):
    pngs.append(save_png(tif, "phase"))
print(f"Saved {len(pngs)} PNG previews next to the GeoTIFFs")

# Bundle every preview into one small zip
zip_path = "Amplitude_Phase_previews.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for p in pngs:
        z.write(p, os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)))
print(f"Zipped to {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

# In Colab, start the download automatically; elsewhere, grab it from the file panel
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab - download the zip from the file browser instead")

Saved 12 PNG previews next to the GeoTIFFs
Zipped to Amplitude_Phase_previews.zip (5.5 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# How many unique track-subswaths, and how many unique bursts per track?
# how many scenes per unique burst?
burstid_unq = sorted(set(burstid))
print(f"{len(burstid_unq)} unique burst IDs out of {num_files} files:")
for b in burstid_unq:
    n_scenes = sum(1 for bid in burstid if bid == b)
    print(f"  {b}: {n_scenes} scene(s)")

3 unique burst IDs out of 6 files:
  t071_151217_iw2: 2 scene(s)
  t071_151218_iw2: 2 scene(s)
  t071_151219_iw2: 2 scene(s)


In [7]:
# How many unique tracks (ignoring subswath)?
tracku = sorted({b.split("_")[0] for b in burstid_unq})
print(f"{len(tracku)} unique track(s): {tracku}")

1 unique track(s): ['t071']


## Generate the Interferogram (TODO — this is the assignment)

For every pair of acquisitions that share the same burst ID:
1. Multiply the complex data of the first (earlier) date by the complex
   conjugate of the second (later) date to get the interferogram.
2. Take the phase angle of the interferogram — this is the phase
   difference / interferometric phase.
3. Save the phase difference as a GeoTIFF so you can view it in QGIS.

Only two lines are needed for you to add for steps 1-2. Everything
else (finding matching pairs, ordering by date, reconstructing each file's
complex data from its saved Amplitude/Phase GeoTIFF, the output folder) is
already provided below.

In [10]:
import gc
import matplotlib
matplotlib.use("Agg")            # draw to file only, no pop-up windows
import matplotlib.pyplot as plt

output_dir_ifg_phs = "./IFG_PHS/"
os.makedirs(output_dir_ifg_phs, exist_ok=True)

PNG_MAX_PIXELS = 2000            # longest side of the PNG preview (keeps files small)


def save_phase_png(phase, png_path, title, transform=None):
    """Save a wrapped-phase preview PNG with a cyclic colour map and colour bar."""
    # Decimate only for the preview image; the GeoTIFF keeps full resolution
    step = max(1, int(np.ceil(max(phase.shape) / PNG_MAX_PIXELS)))
    preview = phase[::step, ::step]

    # Use map coordinates if the transform is an affine (rasterio-style) transform,
    # otherwise fall back to pixel row/column numbers
    extent, xlabel, ylabel = None, "Column (pixels)", "Row (pixels)"
    try:
        a, e, c, f = transform.a, transform.e, transform.c, transform.f
        nrows, ncols = phase.shape
        extent = [c, c + a * ncols, f + e * nrows, f]
        xlabel, ylabel = "Easting (m)", "Northing (m)"
    except AttributeError:
        pass

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(preview, cmap="twilight", vmin=-np.pi, vmax=np.pi,
                   extent=extent, interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Interferometric phase (radians)")
    cbar.set_ticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi])
    cbar.set_ticklabels(["−π", "−π/2", "0", "π/2", "π"])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(png_path, dpi=150)
    plt.close(fig)               # free memory before the next burst


for k, burst in enumerate(burstid_unq, start=1):
    # find indices of files matching this burst id
    ind = [i for i, b in enumerate(burstid) if burst in b]

    # make sure date1 is earlier than date2
    if datestring[ind[0]] > datestring[ind[1]]:
        ind[0], ind[1] = ind[1], ind[0]

    # reconstruct each file's complex data from its saved Amplitude/Phase GeoTIFFs
    C1 = load_complex_from_geotiffs(
        os.path.join(output_dir_amp, f"A{ind[0] + 1}_{datestring[ind[0]]}.tif"),
        os.path.join(output_dir_phs, f"PHS{ind[0] + 1}_{datestring[ind[0]]}.tif"),
    )
    C2 = load_complex_from_geotiffs(
        os.path.join(output_dir_amp, f"A{ind[1] + 1}_{datestring[ind[1]]}.tif"),
        os.path.join(output_dir_phs, f"PHS{ind[1] + 1}_{datestring[ind[1]]}.tif"),
    )

    # TODO: multiply the complex number of the first date (C1) by the
    # complex conjugate of the second date (C2) to get the interferogram
    IFG = None #TO FILL IN, use Numpy

    # TODO: take the phase angle of the interferogram
    IFG_PHS = None #TO FILL IN, use Numpy

    if IFG is None or IFG_PHS is None:
        raise NotImplementedError("Error: go back to fill up IFG and IFG_PHS above, then re-run this cell.")

    # Full-resolution GeoTIFF (for QGIS)
    tif_path = os.path.join(output_dir_ifg_phs, f"Phase{k}.tif")
    write_geotiff(tif_path, IFG_PHS.astype(np.float32), R[ind[0]], epsg[ind[0]])

    # Lightweight PNG preview (for anyone who can't download the GeoTIFFs)
    png_path = os.path.join(output_dir_ifg_phs, f"Phase{k}.png")
    save_phase_png(
        IFG_PHS, png_path,
        title=f"Interferogram phase, burst {burst}\n{datestring[ind[0]]} → {datestring[ind[1]]}",
        transform=R[ind[0]],
    )
    print(f"saved {tif_path} and {png_path}")

    del C1, C2, IFG, IFG_PHS

gc.collect()
print("Done - check ./IFG_PHS/ for Phase*.tif (GeoTIFF) and Phase*.png (preview)")

saved ./IFG_PHS/Phase1.tif and ./IFG_PHS/Phase1.png
saved ./IFG_PHS/Phase2.tif and ./IFG_PHS/Phase2.png
saved ./IFG_PHS/Phase3.tif and ./IFG_PHS/Phase3.png
Done - check ./IFG_PHS/ for Phase*.tif (GeoTIFF) and Phase*.png (preview)


## Download the interferogram phase GeoTIFFs to view in QGIS

Run this once your `IFG_PHS` loop above is filled in and has produced files
in `./IFG_PHS/`.

In [ ]:
import shutil
shutil.make_archive("IFG_PHS", "zip", "IFG_PHS")
from google.colab import files
files.download("IFG_PHS.zip")

## 4. Calculate Correlation Co-efficient

`make_cc` streams the two Amplitude/Phase GeoTIFF pairs off disk in row
chunks and writes `CC.tif` directly — it never holds the full-resolution
rasters in memory (reconstructing `file1`/`file2` in full first, the way the
interferogram section does, is what tends to run out of RAM on a full-size
burst). If this still crashes, lower `rows_per_chunk` in the call below —
e.g. to 50.

In [11]:
# Calculate Correlation Coefficient (given — run this section as-is)
# using burst ID, e.g. "t071_151218_iw2"
target_burst = "t071_151218_iw2"
ind = [i for i, b in enumerate(burstid) if target_burst in b]

# specifying window size. 11x5 meaning 55 pixels around a centred pixel.
# do not change.
wx = 11
wy = 5

cc_file = "CC.tif"  # output file name

# calculates the correlation coefficient for the whole burst (this may take a
# few minutes) and writes it straight to cc_file
make_cc(
    os.path.join(output_dir_amp, f"A{ind[0] + 1}_{datestring[ind[0]]}.tif"),
    os.path.join(output_dir_phs, f"PHS{ind[0] + 1}_{datestring[ind[0]]}.tif"),
    os.path.join(output_dir_amp, f"A{ind[1] + 1}_{datestring[ind[1]]}.tif"),
    os.path.join(output_dir_phs, f"PHS{ind[1] + 1}_{datestring[ind[1]]}.tif"),
    wx, wy, cc_file, R[ind[0]], epsg[ind[0]],
)

print("Finished creating Correlation Coefficient TIFF File (CC.tif)")

Completed - no data written
Finished creating Correlation Coefficient TIFF File (CC.tif)


In [ ]:
from google.colab import files
files.download("CC.tif")

## 5. Crop rasters and threshold in QGIS

Do this part in QGIS on your computer (not in this notebook):

1. Load `CC.tif` (downloaded above) and `20190716.wprobs.geo.tif` into QGIS.
2. Crop both to the same extent: **Raster → Extraction → Clip Raster by
   Extent**, using extent
   `437430.7221,462557.7093,3945328.2161,3966344.5672 [EPSG:32611]`. Save
   the outputs as e.g. `cc_crop.tif` and `wprobs_crop.tif`.
3. Threshold `cc_crop.tif` , where you can balance the number of
   True Positive / False Positive / True Negative / False Negative signals.


We will be comparing the damage probability map from our lab (20190716.wprobs.geo.tif) to the Correlation coefficient calculated.

In [ ]:
from google.colab import files #UPLOAD your cropped tif files here
uploaded = files.upload()  # select cc_crop.tif and wprobs_crop.tif

## 6. Calculate Confusion Matrix

In [ ]:
# Calculating Confusion Matrix
# rename the filenames below to whatever you named your QGIS-cropped files that you uploaded in the cell above
with rasterio.open("") as src: #RENAME the " " to whatever you named it
    wprobs_matrix = src.read(1)
with rasterio.open("") as src: #RENAME the " " to whatever you named it
    cc_matrix = src.read(1)

# The thresholds have been set, chosen from thresholding CC.tif against
# 20190716.wprobs.geo.tif in QGIS (see the tutorial handout, Section 5).
# wprobs values ABOVE wprobs_threshold become 1 (positive).
# Try different thresholding values in QGIS, these are a few starting values.

wprobs_threshold = 0.95
# cc values BELOW cc_threshold become 1 (positive).

cc_threshold = 0.5   # PLACEHOLDER, YOU CAN CHANGE THE VALUES.

TP, TN, FP, FN = make_confusion_matrix(wprobs_matrix, cc_matrix, wprobs_threshold, cc_threshold)
del wprobs_matrix, cc_matrix  # not needed again once we have TP/TN/FP/FN

print("Finished Calculating Confusion Matrix")
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

## Calculating Statistics Using TP, TN, FP, FN (TODO — fill in the equations)

The cell below already has the structure written for you  - you only need to fill
in each equation itself, using `TP`, `TN`, `FP`, `FN`:
- `precision = TP / (TP + FP)`
- `recall = TP / (TP + FN)`
- `F1 = 2 * (precision * recall) / (precision + recall)`
- `accuracy = (TP + TN) / (TP + TN + FP + FN)`

In [ ]:
# TODO: fill in each equation below using TP, TN, FP, FN (see the markdown
# cell above for the formulas). Everything else - the division-by-zero
# guards and the prints - is already written for you.

if TP + FP == 0:
    precision = float("nan")
else:
    precision = None  # TO FILL IN

if TP + FN == 0:
    recall = float("nan")
else:
    recall = None  # TO FILL IN

if precision != precision or recall != recall or (precision + recall) == 0:
    F1 = float("nan")
else:
    F1 = None  # TO FILL IN

total = TP + TN + FP + FN
if total == 0:
    accuracy = float("nan")
else:
    accuracy = None  # TO FILL IN

print(f"True Positives (TP): {TP}")
print(f"True Negatives (TN): {TN}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {F1}")
print(f"Overall Accuracy: {accuracy}")